# OpenArm 2.0 Phase 2 — Continue in Colab

This notebook continues the Phase 2 repo workflow in Colab.

It assumes you have one or both files:

```text
openarm-data-pipeline-phase2.zip
openarm_phase1_outputs.zip
```

The workflow:

1. Upload/unzip the Phase 2 project.
2. Optionally import Phase 1 outputs.
3. Install requirements.
4. Run the repo scripts on `lerobot/libero_object_image`.
5. Generate audit plots, curated manifest, visualizations, and a Phase 2 results write-up.
6. Zip the final repo for GitHub submission.

Default dataset:

```text
lerobot/libero_object_image
```


In [ ]:
# Optional but useful in Colab
from pathlib import Path
import os, json, zipfile, shutil, textwrap, subprocess, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

WORKDIR = Path("/content")
PROJECT_ZIP = WORKDIR / "openarm-data-pipeline-phase2.zip"
PHASE1_ZIP = WORKDIR / "openarm_phase1_outputs.zip"
PROJECT_DIR = WORKDIR / "openarm-data-pipeline"

print("Working directory:", WORKDIR)


## 1. Upload Phase 2 project zip and Phase 1 outputs zip

Run this cell if the files are not already in `/content`.

Upload:

```text
openarm-data-pipeline-phase2.zip
openarm_phase1_outputs.zip
```


In [ ]:
from google.colab import files

if not PROJECT_ZIP.exists() or not PHASE1_ZIP.exists():
    print("Upload openarm-data-pipeline-phase2.zip and/or openarm_phase1_outputs.zip")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Both zip files already exist in /content.")


## 2. Unzip the Phase 2 project


In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_ZIP.exists():
    raise FileNotFoundError("Missing /content/openarm-data-pipeline-phase2.zip. Upload it first.")

with zipfile.ZipFile(PROJECT_ZIP, "r") as zf:
    zf.extractall(WORKDIR)

assert PROJECT_DIR.exists(), f"Expected project directory at {PROJECT_DIR}"

print("Project extracted:", PROJECT_DIR)
print("Top-level files:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" ", p.name)


## 3. Import previous Phase 1 outputs


In [ ]:
phase1_out = PROJECT_DIR / "outputs" / "phase1"
phase1_out.mkdir(parents=True, exist_ok=True)

if PHASE1_ZIP.exists():
    tmp_phase1 = WORKDIR / "phase1_unzipped"
    if tmp_phase1.exists():
        shutil.rmtree(tmp_phase1)
    tmp_phase1.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(PHASE1_ZIP, "r") as zf:
        zf.extractall(tmp_phase1)

    # Copy files from any nested openarm_phase1_outputs directory.
    candidates = list(tmp_phase1.rglob("phase1_audit_summary.csv"))
    if candidates:
        src_dir = candidates[0].parent
        for f in src_dir.iterdir():
            if f.is_file():
                shutil.copy(f, phase1_out / f.name)
        print("Copied Phase 1 outputs from:", src_dir)
    else:
        print("No phase1_audit_summary.csv found inside Phase 1 zip.")
else:
    print("No Phase 1 output zip found. Skipping import.")

print("Phase 1 output directory:", phase1_out)
print([p.name for p in phase1_out.iterdir()])


## 4. Review Phase 1 result summary


In [ ]:
summary_path = phase1_out / "phase1_audit_summary.csv"

if summary_path.exists():
    phase1_df = pd.read_csv(summary_path)
    display(phase1_df.head())
    display(phase1_df.describe(include="all"))

    print("Kept candidates:", phase1_df["keep_candidate"].sum(), "/", len(phase1_df))
    print("Mean episode length:", phase1_df["num_steps"].mean())
    print("Max joint velocity range:", phase1_df["max_abs_joint_velocity"].min(), "to", phase1_df["max_abs_joint_velocity"].max())
    print("Mean bad frame ratio:", phase1_df["bad_frame_ratio"].mean())
else:
    print("No previous Phase 1 summary found.")


## 5. Install requirements

This may take several minutes the first time because `lerobot` and its dependencies need to be installed.


In [ ]:
%cd /content/openarm-data-pipeline

!python -m pip install -q -U pip
!pip install -q -r requirements.txt

print("Installed requirements.")


## 6. Inspect the selected LeRobot dataset


In [ ]:
%cd /content/openarm-data-pipeline

!python scripts/inspect_dataset.py --repo-id lerobot/libero_object_image


## 7. Configure Phase 2 audit

The default config already uses:

```yaml
repo_id: "lerobot/libero_object_image"
wrist_image_key: "observation.images.wrist_image"
```

This cell creates a Colab-friendly config. Start with 50 episodes for speed. Later set `max_episodes: null` for a full run.


In [ ]:
%%writefile configs/curation_colab.yaml
repo_id: "lerobot/libero_object_image"

state_key: null
action_key: null
timestamp_key: null
wrist_image_key: "observation.images.wrist_image"

# Start smaller in Colab. Change to null for full dataset.
max_episodes: 50

# Teleoperation filters
min_episode_len: 20
max_joint_velocity: 20.0
max_timestamp_gap_ratio: 3.0

# Wrist / egocentric video filters
# These are starter thresholds. Use score plots to calibrate.
blur_threshold: 30.0
exposure_threshold: 0.50
frozen_frame_threshold: 0.5
max_bad_frame_ratio: 0.25
video_frame_stride: 5

# Curation
smooth_window: 5
drop_bad_frames_with_mask: true

output_dir: "outputs"


## 8. Run Phase 2 audit


In [ ]:
%cd /content/openarm-data-pipeline

!python scripts/audit_dataset.py --config configs/curation_colab.yaml


## 9. Load and interpret audit outputs


In [ ]:
output_dir = Path("/content/openarm-data-pipeline/outputs")

audit_summary_path = output_dir / "audit_summary.csv"
audit_metadata_path = output_dir / "audit_metadata.json"

audit_df = pd.read_csv(audit_summary_path)
display(audit_df.head())
display(audit_df.describe(include="all"))

with open(audit_metadata_path, "r") as f:
    metadata = json.load(f)

print("Metadata:")
print(json.dumps(metadata, indent=2)[:3000])

print("\nCandidate episodes kept by audit:", int(audit_df["keep_candidate"].sum()), "/", len(audit_df))
print("Unique filter reasons:")
display(audit_df["filter_reasons"].value_counts(dropna=False))


## 10. Show audit plots


In [ ]:
fig_dir = Path("/content/openarm-data-pipeline/outputs/figures")
for path in sorted(fig_dir.glob("*.png")):
    print(path.name)
    img = plt.imread(path)
    plt.figure(figsize=(8, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


## 11. Optional: calibrate video thresholds

If many wrist-camera frames are falsely rejected, inspect the score plots and update `configs/curation_colab.yaml`.

A typical interpretation:

- If blur scores are mostly below threshold but frames look visually fine, lower `blur_threshold`.
- If many frames are marked `bad_exposure` due to dark simulation backgrounds, raise `exposure_threshold`.
- If adjacent frame differences are low because the robot pauses, lower `frozen_frame_threshold` or only apply frozen-frame filtering across longer windows.


In [ ]:
# Quick rejection reason analysis
report_path = output_dir / "audit_report.json"

with open(report_path, "r") as f:
    reports = json.load(f)

from collections import Counter
reason_counts = Counter()

blur_scores, exposure_scores, frame_diffs = [], [], []

for r in reports:
    for fr in r["egocentric_video_report"]["frame_reports"]:
        for reason in fr["reasons"]:
            reason_counts[reason] += 1
        blur_scores.append(fr["blur_score"])
        exposure_scores.append(fr["exposure_score"])
        if fr["frame_difference"] is not None:
            frame_diffs.append(fr["frame_difference"])

print("Video rejection reason counts:", reason_counts)

def stats(name, values):
    if not values:
        print(name, "no values")
        return
    arr = np.asarray(values)
    print(f"{name}: min={arr.min():.4f}, mean={arr.mean():.4f}, median={np.median(arr):.4f}, max={arr.max():.4f}")

stats("Blur", blur_scores)
stats("Exposure", exposure_scores)
stats("Frame diff", frame_diffs)


## 12. Visualize wrist-camera frames


In [ ]:
%cd /content/openarm-data-pipeline

!python scripts/visualize_episode.py --config configs/curation_colab.yaml --episode-id 0 --num-frames 6


In [ ]:
vis_paths = sorted(Path("/content/openarm-data-pipeline/outputs/figures").glob("*wrist_frames.png"))
for path in vis_paths[-3:]:
    print(path.name)
    img = plt.imread(path)
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


## 13. Run curation


In [ ]:
%cd /content/openarm-data-pipeline

!python scripts/curate_dataset.py --config configs/curation_colab.yaml


## 14. Inspect curated manifest


In [ ]:
manifest_path = Path("/content/openarm-data-pipeline/outputs/curated_manifest.json")

with open(manifest_path, "r") as f:
    manifest = json.load(f)

manifest_df = pd.DataFrame([
    {
        "episode_id": m["episode_id"],
        "kept": m["kept"],
        "drop_reasons": ", ".join(m["drop_reasons"]),
        "num_original_steps": m["num_original_steps"],
        "num_valid_aligned_steps": m["num_valid_aligned_steps"],
        "array_path": m.get("array_path", ""),
    }
    for m in manifest
])

display(manifest_df.head())
display(manifest_df["kept"].value_counts())

print("Curated array files:", len(list(Path("/content/openarm-data-pipeline/outputs/curated_arrays").glob("*.npz"))))


## 15. Inspect one curated episode array


In [ ]:
curated_files = sorted(Path("/content/openarm-data-pipeline/outputs/curated_arrays").glob("*.npz"))

if curated_files:
    sample_npz = curated_files[0]
    data = np.load(sample_npz)
    print("Sample curated file:", sample_npz)
    print("Keys:", data.files)
    for k in data.files:
        print(k, data[k].shape, data[k].dtype)

    states = data["states"]
    states_smooth = data["states_smooth"]

    plt.figure(figsize=(8, 4))
    dim = 0
    plt.plot(states[:, dim], label="raw state dim 0")
    plt.plot(states_smooth[:, dim], label="smoothed state dim 0")
    plt.title("Raw vs Smoothed State Trajectory")
    plt.xlabel("Curated timestep")
    plt.ylabel("State value")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No curated arrays found. Check filter thresholds or audit results.")


## 16. Export Label Studio task stubs


In [ ]:
%cd /content/openarm-data-pipeline

!python scripts/export_label_studio_tasks.py \
  --config configs/curation_colab.yaml \
  --manifest outputs/curated_manifest.json \
  --out outputs/label_studio_tasks.json

print("Preview:")
with open("outputs/label_studio_tasks.json", "r") as f:
    tasks = json.load(f)
print(json.dumps(tasks[:3], indent=2))


## 17. Generate Phase 2 results write-up


In [ ]:
result_md = Path("/content/openarm-data-pipeline/docs/RESULTS_PHASE2.md")

num_audited = len(audit_df)
num_keep = int(audit_df["keep_candidate"].sum())
mean_len = float(audit_df["num_steps"].mean())
min_len = int(audit_df["num_steps"].min())
max_len = int(audit_df["num_steps"].max())
max_vel_min = float(audit_df["max_abs_joint_velocity"].min())
max_vel_max = float(audit_df["max_abs_joint_velocity"].max())
mean_bad = float(audit_df["bad_frame_ratio"].mean()) if audit_df["bad_frame_ratio"].notna().any() else None

filter_counts = audit_df["filter_reasons"].value_counts(dropna=False).to_dict()

content = f'''# Phase 2 Results

## Dataset

- Dataset: `lerobot/libero_object_image`
- Audited episodes: {num_audited}
- Selected wrist/egocentric stream: `{metadata["keys"]["wrist_image_key"]}`
- State key: `{metadata["keys"]["state_key"]}`
- Action key: `{metadata["keys"]["action_key"]}`
- Timestamp key: `{metadata["keys"]["timestamp_key"]}`

## Teleoperation audit

The audited episodes have trajectory lengths between {min_len} and {max_len} frames, with an average length of {mean_len:.2f} frames.

The maximum absolute joint/state velocity across episodes ranged from {max_vel_min:.4f} to {max_vel_max:.4f} under the current finite-difference estimate. The audit also checks for NaN/Inf states and timestamp gaps.

## Egocentric / wrist-camera audit

The wrist-camera audit sampled frames with stride `{metadata["thresholds"]["video_frame_stride"]}` and computed:

- blur score using variance of Laplacian
- exposure saturation ratio
- adjacent-frame difference for frozen-frame detection

Mean bad-frame ratio: {mean_bad}

Video rejection reason counts:

```text
{metadata.get("video_reason_counts", {})}
```

## Curation result

Candidate episodes kept by the current filters: {num_keep} / {num_audited}

Filter reason counts:

```text
{filter_counts}
```

The curation step saves a manifest with `original_indices` and `valid_alignment_mask`, so downstream policy training can preserve the mapping between robot state, action, timestamp, and wrist-camera frame.

## Interpretation

Teleoperation filtering is mostly numerical and kinematic: short episodes, invalid states, timestamp gaps, and unrealistic velocity spikes.

Egocentric filtering is perceptual and threshold-sensitive. Blur/exposure/frozen-frame thresholds should be calibrated using score distributions and manual visualization before dropping full episodes.

## Next steps

- run the audit on all episodes by setting `max_episodes: null`
- manually inspect frames near the blur/exposure thresholds
- export real video clips for Label Studio rather than task stubs
- add object visibility and task-success labels
- train a small wrist-camera success detector
'''

result_md.write_text(content, encoding="utf-8")
print(result_md.read_text())


## 18. Zip final Phase 2 repo for download


In [ ]:
%cd /content

FINAL_ZIP = Path("/content/openarm-data-pipeline-final-colab.zip")
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

shutil.make_archive(str(FINAL_ZIP).replace(".zip", ""), "zip", root_dir="/content", base_dir="openarm-data-pipeline")

print("Created:", FINAL_ZIP)
print("Size MB:", FINAL_ZIP.stat().st_size / 1e6)

from google.colab import files
files.download(str(FINAL_ZIP))
